# Web search tool (built-in)

This notebook mirrors the course pattern: load the project root on `sys.path`, configure the Anthropic client, and call the **server-side** `web_search` tool (`web_search_20250305`). The API runs searches and returns a grounded reply—see [Web search tool](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/web-search-tool).

Implementation lives in `utils/web_search.py`; optional prompt snippets are in `prompts/web_search_prompts.py`.

In [ ]:
import os
import sys

# Resolve repo root (folder that contains `utils/`), works from notebooks/ or notebooks/tool_use/
_repo = os.getcwd()
while _repo and not os.path.isdir(os.path.join(_repo, "utils")):
    _parent = os.path.dirname(_repo)
    if _parent == _repo:
        raise RuntimeError("Could not find project root (directory containing utils/)")
    _repo = _parent
sys.path.insert(0, _repo)

from dotenv import load_dotenv

load_dotenv()

In [ ]:
from prompts.web_search_prompts import (
    SCHOLAR_FINANCE_RESEARCH_SYSTEM,
    WEB_SEARCH_SYSTEM_GROUNDED,
    scholar_latest_financial_models_user,
)
from utils.web_search import (
    WebSearchClient,
    WebSearchToolConfig,
    text_from_message,
)

api_key = os.getenv("ANTHROPIC_API_KEY")
if api_key is None:
    raise ValueError("ANTHROPIC_API_KEY not found — add it to a .env file in the repo root.")

model = os.getenv("ANTHROPIC_WEB_SEARCH_MODEL", "claude-sonnet-4-5")

# Same shape as `006_web_search_complete.ipynb`: restrict to NIH for health-focused queries
web_search_config = WebSearchToolConfig(
    max_uses=5,
    allowed_domains=["nih.gov"],
)

client = WebSearchClient(
    api_key=api_key,
    model=model,
    default_tool_config=web_search_config,
)

In [ ]:
user_prompt = """
What's the best exercise for gaining leg muscle?
""".strip()

response = client.ask(
    user_prompt,
    system=WEB_SEARCH_SYSTEM_GROUNDED,
    max_tokens=1000,
    temperature=1.0,
)

response

In [ ]:
print(text_from_message(response))

## Google Scholar: latest financial models

Restrict web search to **`scholar.google.com`** so results come from Google Scholar. The same `WebSearchClient` is reused; **`tool_config`** overrides the NIH-only default for this query only.

In [ ]:
scholar_tool_config = WebSearchToolConfig(
    max_uses=5,
    allowed_domains=["scholar.google.com"],
)

response_scholar = client.ask(
    scholar_latest_financial_models_user(),
    system=SCHOLAR_FINANCE_RESEARCH_SYSTEM,
    tool_config=scholar_tool_config,
    max_tokens=2000,
    temperature=0.4,
)

response_scholar

In [ ]:
print(text_from_message(response_scholar))